# Category Diagnostic Notebook — All Categories, No Pre-Selection

Standalone from `data_preprocessing_and_baseline_model.ipynb` — does not
read or write any of that notebook's files, and does not modify it.
Purpose: produce the evidence needed to decide `CLEARED_CATEGORIES` for
the baseline notebook, rather than assuming a prior cycle's selection
still holds. Runs every check across **all** categories present in the
data, not a pre-filtered subset. Per-product forecasting is out of
scope here — this notebook only informs category-aggregate decisions.

Outputs are written with a run-date suffix so successive diagnostic
runs (as the commodity filter evolves, as more days accumulate) don't
overwrite each other.


## 0. Config

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from datetime import date
import os
import sys
from pathlib import Path

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "vendor" / "pipeline" / "utils").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("vendor/pipeline/utils not found; run `git submodule update --init --recursive`")

sys.path.insert(0, str(REPO_ROOT / "vendor" / "pipeline"))

from utils.commodity_filters import is_valid_commodity, get_commodity_filter_reason

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

RUN_DATE = date.today().isoformat()  # stamps every output file, e.g. 2026-10-17
OUTDIR = "data/output/diagnostics_output/diagnostics_output_4m"
os.makedirs(OUTDIR, exist_ok=True)

MIN_DAYS_PRESENT_RATIOS = [0.6, 0.8, 0.95]
WINDOW_SPLITS = 3


## 1. Load products, apply commodity filter

In [ ]:
products_raw = pd.read_csv("data/raw/products_rows.csv")

products_raw["passes_filter"] = products_raw.apply(
    lambda row: is_valid_commodity(row["name"], row["category"]),
    axis=1,
)
removed = products_raw[~products_raw["passes_filter"]]
products = products_raw[products_raw["passes_filter"]].reset_index(drop=True)

print(f"products_rows.csv: {len(products_raw)} | passed filter: {len(products)} | removed: {len(removed)}")


### Filler

In [ ]:
TEA_DIR = "result/investigations/tea_contamination"
removed[removed['category'] == 'tea']['name'].to_csv(f"{TEA_DIR}/tea_excluded.csv")

In [ ]:
products[products['category'] == 'tea']['name'].to_csv(f"{TEA_DIR}/tea_kept.csv")

In [ ]:
kept = pd.read_csv(f"{TEA_DIR}/tea_kept.csv")  # or whatever you named this export
names = kept["name"].str.lower()

# Expand the positive-match term list with known brand/product-line names
tea_terms = r"tea|chai|danedar|tapal|lipton|patti|chaye|karak|tezdum|family mixture|dahleez|bengal|zordar|harmony|yellow label"

names = kept["name"].str.lower()
has_tea_term = names.str.contains(tea_terms, na=False, regex=True)

print(f"Contain a tea-related term (expanded): {has_tea_term.sum()} ({has_tea_term.mean()*100:.1f}%)")
print(f"Still no match (likely genuine junk): {(~has_tea_term).sum()}")
print()
print(kept.loc[~has_tea_term, "name"].sample(30, random_state=1).to_string(index=False))

#filler End

## 2. Load prices, merge onto filtered products

In [ ]:
prices = pd.read_csv("data/raw/product_prices_rows.csv")
prices = prices[prices["pipeline_version"] == "2.0.0"].reset_index(drop=True)

merged = prices.merge(
    products[["id", "name", "category", "source", "url"]],
    left_on="product_id", right_on="id", how="inner",
)
print(f"product_prices_rows.csv (pipeline_version=2.0.0): {len(prices)} | merged onto filtered products: {len(merged)}")


## 3. Base prep — discount flags, calendar fields, Daraz-only

In [ ]:
df = merged[[
    "scraped_at", "run_id", "name", "category", "source", "url",
    "unit_price", "unit_type", "unit_value",
    "final_price", "original_price", "sale_price", "rating", "review_count",
]].copy()

df["scraped_at"] = pd.to_datetime(df["scraped_at"])
df["hour"] = df["scraped_at"].dt.hour
df["is_discounted"] = df["sale_price"].notna() & (df["sale_price"] < df["original_price"])
df["discount_pct"] = np.where(
    df["is_discounted"],
    (df["original_price"] - df["sale_price"]) / df["original_price"] * 100,
    0.0,
)

daraz_df = df[df["source"] == "daraz"].sort_values(["url", "scraped_at"]).reset_index(drop=True)
print(f"{len(daraz_df)} rows, {daraz_df['url'].nunique()} products, {daraz_df['category'].nunique()} categories")
print(sorted(daraz_df["category"].unique()))


## 4. Collapse to daily (modeling grain)

One row per product per day. `unit_price` takes the day's median;
discount/rating fields take the max across whichever session(s) exist
that day. Same collapse logic as the baseline notebook — no reason to
compute this differently in two places.


In [ ]:
d = daraz_df.copy()
d["obs_date"] = d["scraped_at"].dt.date

daraz_daily = d.groupby(["url", "obs_date"]).agg(
    category=("category", "first"), source=("source", "first"), name=("name", "first"),
    unit_price=("unit_price", "median"), unit_type=("unit_type", "first"),
    is_discounted=("is_discounted", "max"), discount_pct=("discount_pct", "max"),
    rating=("rating", "max"), review_count=("review_count", "max"),
).reset_index()

daraz_daily["obs_date"] = pd.to_datetime(daraz_daily["obs_date"])
daraz_daily = daraz_daily.sort_values(["url", "obs_date"]).reset_index(drop=True)

print(f"{len(daraz_daily)} product-days, {daraz_daily['url'].nunique()} products")
print(f"Date range: {daraz_daily['obs_date'].min().date()} to {daraz_daily['obs_date'].max().date()}")


## 5. Per-category panel stats

No category filter applied — every category present gets a row.


In [ ]:
def panel_stats(frame):
    g = frame.groupby("url")["unit_price"]
    cv = (g.std() / g.mean()).fillna(0)
    obs = frame.groupby("url").size()
    return pd.Series({
        "n_products": frame["url"].nunique(),
        "pct_flat": (cv == 0).mean() * 100,
        "median_cv": cv.median(),
        "median_obs": obs.median(),
        "min_obs": obs.min(),
        "max_obs": obs.max(),
    })

panel_table = daraz_daily.groupby("category").apply(panel_stats).round(2)
panel_table = panel_table.sort_values("pct_flat")
print(panel_table)


## 6. Stability check — all-listings median vs fixed-URL (balanced panel) median

Compares the daily category median computed from every listing that
day against the daily median computed only from URLs present on at
least `ratio` of all days in the window. If the all-listings series is
much noisier than the balanced-panel series, the noise is coming from
*which listings happen to appear*, not from real price movement — this
is a direct test of the mechanism suspected (not assumed) behind any
category's instability. Run at several ratio thresholds since the
right balance point likely isn't uniform across categories with very
different product counts.


In [ ]:
def stability_check(frame, category, ratio):
    cat_df = frame[frame.category == category]
    total_days = cat_df["obs_date"].nunique()

    days_present = cat_df.groupby("url")["obs_date"].nunique()
    balanced_urls = days_present[days_present >= total_days * ratio].index

    all_median = cat_df.groupby("obs_date")["unit_price"].median()
    balanced_median = (
        cat_df[cat_df.url.isin(balanced_urls)]
        .groupby("obs_date")["unit_price"]
        .median()
    )

    all_std = all_median.pct_change().std()
    balanced_std = balanced_median.pct_change().std()
    ratio_val = all_std / balanced_std if balanced_std and not np.isnan(balanced_std) else np.nan

    return {
        "category": category,
        "ratio_threshold": ratio,
        "n_balanced_urls": len(balanced_urls),
        "n_total_urls": cat_df["url"].nunique(),
        "all_listings_std": all_std,
        "balanced_panel_std": balanced_std,
        "noise_ratio": ratio_val,
    }

stability_rows = []
for category in sorted(daraz_daily["category"].unique()):
    for ratio in MIN_DAYS_PRESENT_RATIOS:
        stability_rows.append(stability_check(daraz_daily, category, ratio))

stability_table = pd.DataFrame(stability_rows).round(4)
print(stability_table)


## 7. Category-aggregate volatility — full window + windowed split

Default output includes the multi-way window split, not just the
full-window summary — a full-window-only stat can hide a trend that
only shows up once the window is split (confirmed necessary in a prior
cycle: a category that looked unremarkable on the full window showed a
clear monotonic climb only visible in a 3-way split).


In [ ]:
category_daily = (
    daraz_daily.groupby(["category", "obs_date"])["unit_price"]
    .median()
    .reset_index()
    .sort_values(["category", "obs_date"])
    .reset_index(drop=True)
)

cg = category_daily.groupby("category")["unit_price"]
category_daily["cat_price_change_pct"] = cg.pct_change()

def volatility_table(frame):
    return frame.groupby("category")["cat_price_change_pct"].agg(
        n_days="count", mean="mean", std="std", median="median",
        pct_nonzero=lambda s: (s != 0).mean() * 100,
    ).round(4)

print("=== FULL WINDOW ===")
full_vol = volatility_table(category_daily)
print(full_vol)

mind, maxd = category_daily["obs_date"].min(), category_daily["obs_date"].max()
span = (maxd - mind).days
edges = [mind - pd.Timedelta(days=1)] + [
    mind + pd.Timedelta(days=span * i / WINDOW_SPLITS) for i in range(1, WINDOW_SPLITS + 1)
]
labels = [f"W{i+1}" for i in range(WINDOW_SPLITS)]
category_daily["window"] = pd.cut(category_daily["obs_date"], bins=edges, labels=labels)

windowed_vol = {}
for w in labels:
    sub = category_daily[category_daily["window"] == w]
    print(f"\n=== {w} ({sub['obs_date'].min()} to {sub['obs_date'].max()}) ===")
    wv = volatility_table(sub)
    print(wv)
    windowed_vol[w] = wv


## 8. Price-spread / span-ratio table

In [ ]:
def spread_stats(g):
    s = g.describe(percentiles=[0.1, 0.5, 0.9])
    return pd.Series({
        "n": s["count"],
        "min": s["min"],
        "p10": s["10%"],
        "median": s["50%"],
        "p90": s["90%"],
        "max": s["max"],
        "span_ratio": s["max"] / s["min"] if s["min"] else np.nan,
    })

spread_table = daraz_daily.groupby("category")["unit_price"].apply(spread_stats).unstack().round(2)
spread_table = spread_table.sort_values("span_ratio")
print(spread_table)


## 9. Day/night (session timing) check — all categories

Not assumed to still hold under the new commodity filter — re-run
fresh, same three-part logic as the original check (median gap, paired
Wilcoxon, discount-conditioned), reused rather than rewritten.


In [ ]:
def day_night_median_gap(d):
    x = d.copy()
    x["run_period"] = np.where(x["hour"] < 12, "morning", "evening")
    stats = x.groupby(["category", "run_period"])["unit_price"].agg(["count", "mean", "median", "std"])
    pivot = stats["median"].unstack()
    pct_diff = ((pivot["evening"] - pivot["morning"]) / pivot["morning"] * 100).round(2)
    return stats, pct_diff.sort_values()

median_gap_stats, median_gap_pct = day_night_median_gap(daraz_df)
print(median_gap_stats)
print()
print(median_gap_pct)


In [ ]:
def paired_day_night_check(d):
    x = d.copy()
    x["run_period"] = np.where(x["hour"] < 12, "morning", "evening")
    x["obs_date"] = x["scraped_at"].dt.date

    daily = x.groupby(["url", "category", "obs_date", "run_period"])["unit_price"].median().reset_index()
    wide = daily.pivot_table(index=["url", "category", "obs_date"], columns="run_period", values="unit_price").dropna()
    wide["diff"] = wide["evening"] - wide["morning"]

    print(f"paired product-days: {len(wide)}")
    print(wide.groupby("category")["diff"].agg(count="count", mean="mean", median="median", std="std").round(3))
    print()

    for category, group in wide.groupby("category"):
        if len(group) < 10:
            continue
        stat, p = wilcoxon(group["morning"], group["evening"])
        flag = "  <-- possible effect" if p < 0.05 else ""
        print(f"  {category}: n={len(group)} p={p:.4f}{flag}")

    return wide

paired_wide = paired_day_night_check(daraz_df)


## 10. Synthesis scorecard

One row per category, pulling the key number from each section above.
No verdict computed here — category selection stays a decision made
by reviewing this table, not logic baked into the notebook.


In [ ]:
best_noise_ratio = (
    stability_table.sort_values("ratio_threshold", ascending=False)
    .groupby("category")
    .first()[["noise_ratio", "n_balanced_urls", "ratio_threshold"]]
)

scorecard = panel_table.join(spread_table[["span_ratio"]])
scorecard = scorecard.join(full_vol[["std", "pct_nonzero"]].rename(columns={"std": "full_window_std"}))
scorecard = scorecard.join(best_noise_ratio)
scorecard = scorecard.join(median_gap_pct.rename("day_night_median_gap_pct"))

window_std_cols = {w: windowed_vol[w]["std"] for w in labels}
window_std_df = pd.DataFrame(window_std_cols)
window_std_df.columns = [f"{w}_std" for w in window_std_df.columns]
scorecard = scorecard.join(window_std_df)

scorecard = scorecard.round(4)
print(scorecard)


## 11. Save outputs (dated, so successive runs don't overwrite each other)

In [ ]:
panel_table.to_csv(f"{OUTDIR}/panel_stats_{RUN_DATE}.csv")
stability_table.to_csv(f"{OUTDIR}/stability_check_{RUN_DATE}.csv", index=False)
full_vol.to_csv(f"{OUTDIR}/category_volatility_full_{RUN_DATE}.csv")
for w in labels:
    windowed_vol[w].to_csv(f"{OUTDIR}/category_volatility_{w}_{RUN_DATE}.csv")
spread_table.to_csv(f"{OUTDIR}/price_spread_{RUN_DATE}.csv")
median_gap_stats.to_csv(f"{OUTDIR}/day_night_median_gap_{RUN_DATE}.csv")
paired_wide.to_csv(f"{OUTDIR}/day_night_paired_{RUN_DATE}.csv", index=False)
scorecard.to_csv(f"{OUTDIR}/scorecard_{RUN_DATE}.csv")

print(f"All diagnostic outputs saved to {OUTDIR}/ with suffix _{RUN_DATE}")
